<a href="https://www.kaggle.com/code/avikdas567/wids-global-datathon-2026?scriptVersionId=306936941" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from scipy.stats import rankdata
import warnings
warnings.filterwarnings("ignore")

# Paths

TRAIN_PATH = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/train.csv"
TEST_PATH = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/test.csv"

# Load data

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

ID_COL = "event_id"
TIME_COL = "time_to_hit_hours"
EVENT_COL = "event"

FEATURES = [c for c in train.columns if c not in [ID_COL, TIME_COL, EVENT_COL]]

# Preprocessing

for col in FEATURES:
    if train[col].dtype == "object":
        train[col] = train[col].fillna("missing")
        test[col] = test[col].fillna("missing")
    else:
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col] = test[col].fillna(med)

# Horizons

HORIZONS = [12, 24, 48, 72]

# Survival target builder

def build_target(df, horizon):
    y, mask = [], []
    for t, e in zip(df[TIME_COL], df[EVENT_COL]):
        if e == 1 and t <= horizon:
            y.append(1); mask.append(True)
        elif t > horizon:
            y.append(0); mask.append(True)
        else:
            y.append(0); mask.append(False)
    return np.array(y), np.array(mask)

# LightGBM params

params = {
    "objective": "binary",
    "learning_rate": 0.03,
    "num_leaves": 24,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 3,
    "min_data_in_leaf": 8,
    "lambda_l1": 0.5,
    "lambda_l2": 0.5,
    "verbosity": -1,
    "seed": 42
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Train per horizon

preds = {}

for h in HORIZONS:
    print(f"\nTraining horizon {h}h")

    y, mask = build_target(train, h)
    X = train[FEATURES][mask]
    y = y[mask]

    test_preds = np.zeros(len(test))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        model = lgb.LGBMClassifier(**params, n_estimators=2000)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )

        test_preds += model.predict_proba(test[FEATURES])[:, 1] / kf.n_splits

    preds[h] = test_preds

# Simple weighted blending

final = {}

final[12] = preds[12]
final[24] = 0.7 * preds[24] + 0.3 * preds[12]
final[48] = 0.6 * preds[48] + 0.4 * preds[24]
final[72] = 0.5 * preds[72] + 0.5 * preds[48]

# Calibration

for h in HORIZONS:
    r = rankdata(final[h]) / len(final[h])
    final[h] = 0.8 * final[h] + 0.2 * r

# Submission

submission = pd.DataFrame({
    "event_id": test[ID_COL],
    "prob_12h": final[12],
    "prob_24h": final[24],
    "prob_48h": final[48],
    "prob_72h": final[72],
})

# Enforce monotonicity

submission["prob_24h"] = np.maximum(submission["prob_24h"], submission["prob_12h"])
submission["prob_48h"] = np.maximum(submission["prob_48h"], submission["prob_24h"])
submission["prob_72h"] = np.maximum(submission["prob_72h"], submission["prob_48h"])

# Clip
for col in ["prob_12h","prob_24h","prob_48h","prob_72h"]:
    submission[col] = submission[col].clip(0,1)

# Save

submission.to_csv("submission.csv", index=False)

print("\n'submission.csv' created.\n")
display(submission.head())


Training horizon 12h

Training horizon 24h

Training horizon 48h

Training horizon 72h

'submission.csv' created.



,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.101069,0.123515,0.128666,0.128666
1,13353600,0.565969,0.826683,0.962766,0.962766
2,13942327,0.074802,0.074802,0.099809,0.114150
3,16112781,0.706088,0.866594,0.952652,0.952652
4,17132808,0.163324,0.163324,0.163324,0.163324
